In [7]:
from pathlib import Path
import pandas as pd
import json
from datasets import load_dataset
import ast
import subprocess
import sys

In [8]:
def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError(f"No .git folder found above {start}")

ROOT = find_repo_root()
DATA_DIR = ROOT / "data"
CORPUS_PATH = DATA_DIR / "corpus" / "pooled_corpus.parquet"
SPLITS_DIR = ROOT / "splits"
EVAL_SCRIPT = ROOT / "eval" / "hotpot_evaluate_v1.py"

# Folders end in _DIR, files end in _PATH
GOLD_DIR = ROOT / "results" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)
GOLD_PATH = GOLD_DIR / "pilot_gold.json"
PRED_PATH = GOLD_DIR / "pilot_gold_as_pred.json"

if str(ROOT / "eval") not in sys.path:
    sys.path.insert(0, str(ROOT / "eval"))

from hotpot_evaluate_v1 import exact_match_score, f1_score

In [9]:
# Get Pilot Slice ids for development
with open(SPLITS_DIR / "pilot.json", "r", encoding="utf-8") as file:
    slice_ids = json.load(file)

# Load Dataset for questions
hotpot_ds = load_dataset("hotpotqa/hotpot_qa", "distractor")

# Get questions for the slice ids only,
train_q = hotpot_ds["train"].select_columns(["id", "answer", "supporting_facts"]).to_pandas()
train_q = train_q.set_index("id")
slice_q = train_q.loc[slice_ids]
assert list(slice_q.index) == slice_ids


In [10]:
def gold_converter(q_rows):
    gold_standard = []

    for qid, answer, sf in zip(q_rows.index, q_rows.answer, q_rows.supporting_facts):
        gold_format = {
            "_id": qid, "answer": answer, 
            "supporting_facts": [[title, int(sent_id)] for title, sent_id in zip(sf["title"], sf["sent_id"])]
        }
        gold_standard.append(gold_format)

    return gold_standard

In [11]:
gold = gold_converter(slice_q)

assert len(gold) == 200
assert [g["_id"] for g in gold] == slice_ids
print (f"{gold[0]} --- {slice_q.iloc[0]}")

with open(GOLD_PATH, "w") as f:
    json.dump(gold, f, indent=2)

{'_id': '5abd94d55542992ac4f382d5', 'answer': 'Steve Jobs', 'supporting_facts': [['Susan Barnes (computing)', 2], ['Steve Jobs', 0]]} --- answer                                                     Steve Jobs
supporting_facts    {'title': ['Susan Barnes (computing)', 'Steve ...
Name: 5abd94d55542992ac4f382d5, dtype: object


In [12]:
# Gold answers and supporting facts used as predictions
gold_as_pred = {
    "answer": {g["_id"]: g["answer"] for g in gold},
    "sp": {g["_id"]: g["supporting_facts"] for g in gold},
}
with open(PRED_PATH, "w") as f:
    json.dump(gold_as_pred, f, indent=2)

# Run the official script with this kernel's Python (the nlp-cw env)
result = subprocess.run(
    [sys.executable, str(EVAL_SCRIPT), str(PRED_PATH), str(GOLD_PATH)],
    capture_output=True, text=True, check=True,
)
print(result.stdout)

# The script prints a Python dict; parse it and check every metric is 1.0
metrics = ast.literal_eval(result.stdout.strip().splitlines()[-1])
assert all(v == 1.0 for v in metrics.values()), metrics
print("Gold-as-prediction test passed")

{'em': 1.0, 'f1': 1.0, 'prec': 1.0, 'recall': 1.0, 'sp_em': 1.0, 'sp_f1': 1.0, 'sp_prec': 1.0, 'sp_recall': 1.0, 'joint_em': 1.0, 'joint_f1': 1.0, 'joint_prec': 1.0, 'joint_recall': 1.0}

Gold-as-prediction test passed


## Convert Outputs
- Turn system outputs into the file the official scripts reads.

**Inputs**: 
- answers: {qid: answer}
- sp [optional]: {qid: [[title, sent_id], ...]} $\rightarrow$ used only if we predict supporting facts
- expected_ids: The IDs that should be there
- output path

In [13]:
def write_predictions(answers, expected_ids, output_path, sp=None):
    output_path = Path(output_path)

    # Check expected_ids has no duplicates (a set would hide them)
    if len(set(expected_ids)) != len(expected_ids):
        raise ValueError(f"expected_ids has {len(expected_ids) - len(set(expected_ids))} duplicate IDs")

    # Check coverage: every expected ID answered, no unexpected IDs
    missing_ids = set(expected_ids) - set(answers)
    extra_ids = set(answers) - set(expected_ids)
    if missing_ids or extra_ids:
        raise ValueError(f"{len(missing_ids)} missing, {len(extra_ids)} extra; e.g. missing {sorted(missing_ids)[:5]}, extra {sorted(extra_ids)[:5]}")

    # Check sp only covers expected IDs (otherwise they would be dropped silently)
    sp = sp or {}
    extra_sp_ids = set(sp) - set(expected_ids)
    if extra_sp_ids:
        raise ValueError(f"{len(extra_sp_ids)} sp IDs not in expected_ids, e.g. {sorted(extra_sp_ids)[:5]}")

    # Check types: every answer must be a string
    bad_types = {qid: type(a).__name__ for qid, a in answers.items() if not isinstance(a, str)}
    if bad_types:
        examples = list(bad_types.items())[:5]
        raise TypeError(f"{len(bad_types)} answers are not strings, e.g. {examples}")

    # Build the official format, in expected_ids order
    predictions = {
        "answer": {qid: answers[qid] for qid in expected_ids},
        "sp": {
            qid: [[title, int(sent_id)] for title, sent_id in sp.get(qid, [])] for qid in expected_ids
        },
    }

    # Save
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(predictions, f, indent=2)

    return predictions

In [14]:
def run_official_eval(pred_path, gold_path):
    """Run the official script with this kernel's Python; return its metrics and printed lines."""
    result = subprocess.run(
        [sys.executable, str(EVAL_SCRIPT), str(pred_path), str(gold_path)],
        capture_output=True, text=True, check=True,
    )
    lines = result.stdout.strip().splitlines()
    return ast.literal_eval(lines[-1]), lines[:-1]

gold_answers = {g["_id"]: g["answer"] for g in gold}
gold_sp = {g["_id"]: g["supporting_facts"] for g in gold}

# Test 1: gold answers + gold sp -> every metric 1.0
write_predictions(gold_answers, slice_ids, PRED_PATH, sp=gold_sp)
metrics, printed = run_official_eval(PRED_PATH, GOLD_PATH)
assert all(v == 1.0 for v in metrics.values()), metrics

# Test 2: gold answers, no sp -> em = f1 = 1.0, sp scores 0, no "missing" lines
write_predictions(gold_answers, slice_ids, PRED_PATH)
metrics, printed = run_official_eval(PRED_PATH, GOLD_PATH)
assert metrics["em"] == metrics["f1"] == 1.0 and metrics["sp_f1"] == 0.0, metrics
assert not any("missing" in line for line in printed), printed[:3]

# Test 3: one ID missing -> ValueError before writing
try:
    write_predictions({k: v for k, v in gold_answers.items() if k != slice_ids[0]}, slice_ids, PRED_PATH)
    raise AssertionError("missing ID was not caught")
except ValueError as e:
    print("Test 3 caught:", e)

# Test 4: a None answer -> TypeError
try:
    write_predictions({**gold_answers, slice_ids[0]: None}, slice_ids, PRED_PATH)
    raise AssertionError("None answer was not caught")
except TypeError as e:
    print("Test 4 caught:", e)

print("All write_predictions tests passed")

Test 3 caught: 1 missing, 0 extra; e.g. missing ['5abd94d55542992ac4f382d5'], extra []
Test 4 caught: 1 answers are not strings, e.g. [('5abd94d55542992ac4f382d5', 'NoneType')]
All write_predictions tests passed


In [ ]:
# qid -> question type (bridge or comparison)
train_types = hotpot_ds["train"].select_columns(["id", "type"]).to_pandas()
qid_to_type = dict(zip(train_types.id, train_types.type))

def score_answers(predictions, gold, qit_to_type):
    """Per-question EM/F1 with the official functions, plus averages overall and per type."""
    rows = []
    for g in gold:
        qid = g["_id"]
        pred = predictions["answer"][qid]
        f1, precision, recall = f1_score(pred, g["answer"])
        rows.append({
            "id": qid,
            "type": qid_to_type[qid],
              "pred": pred,
              "gold": g["answer"],
              "em": float(exact_match_score(pred, g["answer"])),
              "f1": f1,
              "precision": precision,
              "recall": recall,
        })

    per_question = pd.DataFrame(rows)
    def summarise(df):
          return {"n": len(df), "em": float(100 * df.em.mean()), "f1": float(100 * df.f1.mean())}
         
    summary = {"overall": summarise(per_question)}
    for qtype, group in per_question.groupby("type"):
        summary[qtype] = summarise(group)
    return per_question, summary

